In [ ]:
### this is to just login into briker terminal . make sure whenever we login for first time 
# place an dummy order get Super token by clicking shit+ ctrl+i --> network --> payload .


import csv
from datetime import datetime as dt_datetime, timedelta
import time
import threading
import logging
import pandas as pd
from NorenRestApiPy.NorenApi import NorenApi
import pyotp
import yaml
import token
from dateutil.relativedelta import relativedelta


class ShoonyaApiPy(NorenApi):
    def __init__(self):
        # super().__init__(host='https://api.shoonya.com/NorenWClientTP/', websocket='wss://api.shoonya.com/NorenWSTP/')
        super().__init__(host='https://trade.shoonya.com/NorenWClientWeb/', websocket='wss://trade.shoonya.com/NorenWSWeb/')



# Initialize API
api = ShoonyaApiPy()

with open('cred.yml') as f:
    cred = yaml.load(f, Loader=yaml.FullLoader)

SuperToken ="389035229569243b6b2d4f412c355b5610a73a3855ba4c119d4c74b7a1b1e257"
userId=cred['user']
Passwrd=cred['pwd']
ret = api.set_session(userId , Passwrd ,SuperToken )


if ret:
    print("Login Successful")
else:
    print("Login Failed")
    exit()

print(ret)



In [ ]:
import pandas as pd
import os

input_csv = r'C:\Users\omkar\Downloads\Backtest bb_blast_sell_Combined.csv'
filtered_csv = r'C:\Users\omkar\Downloads\filtered_stocks.csv'


# ==================== STEP 1: FILTER BACKTEST CSV ====================
def process_and_save(file_path, save_path):
    try:
        df = pd.read_csv(file_path)

        # First column should be date/time
        date_col = df.columns[0]

        # Convert date column safely
        df[date_col] = pd.to_datetime(
            df[date_col].astype(str).str.strip(),
            format='%d-%m-%Y %I:%M %p',
            errors='coerce'
        )

        # Remove invalid date rows
        df = df.dropna(subset=[date_col])

        # Extract time
        df['time_only'] = df[date_col].dt.strftime('%H:%M')

        # Filter required times
        valid_times = ['09:45', '10:00', '10:30', '10:45']
        filtered_df = df[df['time_only'].isin(valid_times)].copy()

        # Keep only timestamps where stocks <= 3
        filtered_df['timestamp_str'] = filtered_df[date_col].dt.strftime('%Y-%m-%d %H:%M:%S')

        counts = filtered_df['timestamp_str'].value_counts()

        valid_timestamps = counts[counts <= 3].index

        filtered_df = filtered_df[
            filtered_df['timestamp_str'].isin(valid_timestamps)
        ].copy()

        # Optional: remove helper columns before saving
        filtered_df = filtered_df.drop(columns=['time_only', 'timestamp_str'])

        # Save filtered CSV
        filtered_df.to_csv(save_path, index=False)

        return filtered_df

    except Exception as e:
        print(f"\n❌ ERROR in filtering: {e}")
        return None


# ==================== STEP 2: EXTRACTION ====================
def extract_from_csv_file(file_path):
    try:
        df = pd.read_csv(file_path)

        stocks_info = []

        for _, row in df.iterrows():
            stocks_info.append(
                (
                    str(row.iloc[0]).strip(),
                    str(row.iloc[1]).strip()
                )
            )

        return stocks_info

    except Exception as e:
        print(f"❌ Error reading CSV: {e}")
        return []


# ==================== STEP 3: RUN FILTER ====================
filtered_df = process_and_save(input_csv, filtered_csv)


# ==================== STEP 4: LOAD FILTERED CSV ====================
stocks_info = []

if os.path.exists(filtered_csv):
    print("\n📁 Loading from filtered CSV...")

    stocks_info = extract_from_csv_file(filtered_csv)

    if stocks_info:
        print(f"✓ Loaded {len(stocks_info)} stocks")
    else:
        print("❌ No data found in filtered CSV")
else:
    print("❌ Filtered CSV not found")


# ==================== FINAL OUTPUT ====================
print("\n" + "=" * 80)
print("📦 FINAL stocks_info")
print("=" * 80)

if stocks_info:
    print("stocks_info = [")
    for dt, sym in stocks_info:
        print(f"    ('{dt}', '{sym}'),")
    print("]")
else:
    print("❌ No stocks found")

print("=" * 80)
print(f"📊 Total stocks: {len(stocks_info)}")
print("=" * 80)


In [ ]:
################################################################################
#################################################################################
# 1

# 1st script to run to get the stocks_info list which will be used in backtesting. 
# You can copy the output list and paste it in the backtesting code. Make sure to have the filtered_stocks.csv file ready with the correct data for this to work.

################################################################################
#################################################################################





import pandas as pd
import os
from io import StringIO

# print("=" * 80)
# print("FULL PIPELINE: FILTER + LIMIT + EXTRACT stocks_info")
# print("=" * 80)
# print()

# 🔹 FILE PATHS
input_csv = r'C:\Users\omkar\Downloads\Backtest bb_blast_sell_Combined.csv'
filtered_csv = r'C:\Users\omkar\Downloads\filtered_stocks.csv'

# ==================== STEP 1: FILTER BACKTEST CSV ====================
def process_and_save(file_path, save_path):
    try:
        df = pd.read_csv(file_path)

        # print("\n📥 Raw Data:")
        # print(df.head())

        # ✅ Convert to datetime (your format)
        date_col = df.columns[0]
        df[date_col] = pd.to_datetime(
            df[date_col].astype(str).str.strip(),
            format='%d-%m-%Y %I:%M %p',
            errors='coerce'
        )

        # Remove invalid rows
        df = df.dropna(subset=[date_col])

        # Extract time
        df['time_only'] = df[date_col].dt.strftime('%H:%M')

        # print("\n🕒 Time distribution:")
        # print(df['time_only'].value_counts())

        # Step 1: Filter required times
        valid_times = ['09:45', '10:00', '10:30', '10:45']
        filtered_df = df[df['time_only'].isin(valid_times)].copy()

        # print("\n✅ After Time Filter:")
        # print(filtered_df)

        # ==================== NEW RULE ====================
        # Keep only timestamps where stocks <= 3
        filtered_df['timestamp_str'] = filtered_df[date_col].dt.strftime('%Y-%m-%d %H:%M:%S')

        counts = filtered_df['timestamp_str'].value_counts()

        # print("\n📊 Stocks per timestamp:")
        # print(counts)

        # Keep only timestamps with <= 3 stocks
        valid_timestamps = counts[counts <= 3].index

        filtered_df = filtered_df[filtered_df['timestamp_str'].isin(valid_timestamps)]

        # print("\n✅ After Applying Max 3 Stocks Rule:")
        # print(filtered_df)

        # Save filtered CSV
        filtered_df.to_csv(save_path, index=False)
        # print(f"\n💾 Saved filtered file: {save_path}")

        return filtered_df

    except Exception as e:
        print(f"\n❌ ERROR in filtering: {e}")
        return None


# ==================== STEP 2: EXTRACTION ====================
def extract_from_csv_file(file_path):
    try:
        df = pd.read_csv(file_path)

        stocks_info = []
        for _, row in df.iterrows():
            stocks_info.append((str(row.iloc[0]).strip(), str(row.iloc[1]).strip()))

        return stocks_info

    except Exception as e:
        print(f"✗ Error reading CSV: {e}")
        return []


# ==================== STEP 3: RUN FILTER ====================
filtered_df = process_and_save(input_csv, filtered_csv)

# ==================== STEP 4: LOAD FILTERED CSV ====================
stocks_info = []

if os.path.exists(filtered_csv):
    print("\n📁 Loading from filtered CSV...")

    stocks_info = extract_from_csv_file(filtered_csv)

    if stocks_info:
        print(f"✓ Loaded {len(stocks_info)} stocks")
    else:
        print("❌ No data found in filtered CSV")
else:
    print("❌ Filtered CSV not found")

# ==================== FINAL OUTPUT ====================
print("\n" + "=" * 80)
print("📦 FINAL stocks_info")
print("=" * 80)

if stocks_info:
    print("stocks_info = [")
    for dt, sym in stocks_info:
        print(f"    ('{dt}', '{sym}'),")
    print("]")
else:
    print("❌ No stocks found")

print("=" * 80)
print(f"📊 Total stocks: {len(stocks_info)}")
print("=" * 80)

In [ ]:

######################################################################################
#2 
        # This script is used for downloading historical data for a list of stocks using 
        # the Shoonya API.
        #it need Stocks_info which needs to be updated from other script 

#######################################################################################
    


import csv
from datetime import datetime, timedelta
import time
import threading
import logging
import pandas as pd
from NorenRestApiPy.NorenApi import NorenApi
import pyotp
import yaml
import token
from dateutil.relativedelta import relativedelta


print("=" * 80)
print("COMPLETE WORKFLOW: LOGIN → EXTRACT SYMBOLS → DOWNLOAD DATA")
print("=" * 80)
print()

# ==================== STEP 1: EXTRACT SYMBOLS WITH -EQ SUFFIX ====================
print("📊 STEP 1: EXTRACTING SYMBOLS FROM stocks_info")
print("=" * 80)

symbols_array = []

stocks_info = [
    ('2026-05-19 10:00:00', 'AEGISLOG'),
    ('2026-05-19 10:00:00', 'INTELLECT'),
    ('2026-05-19 10:00:00', 'SWIGGY'),
    ('2026-05-19 10:30:00', 'DEEPAKFERT'),
    ('2026-05-19 10:30:00', 'KPIL'),
    ('2026-05-19 10:45:00', 'STARHEALTH'),
    ('2026-05-21 10:00:00', 'GODFRYPHLP'),
    ('2026-05-21 10:00:00', 'GRAPHITE'),
    ('2026-05-21 10:00:00', 'DATAPATTNS'),
    ('2026-05-21 10:30:00', 'ADANIPOWER'),
    ('2026-05-22 09:45:00', 'DALBHARAT'),
    ('2026-05-22 10:00:00', 'TRITURBINE'),
    ('2026-05-22 10:00:00', 'SBILIFE'),
    ('2026-05-27 10:30:00', 'THERMAX'),
    ('2026-05-27 10:30:00', 'AIAENG'),
    ('2026-05-27 10:30:00', 'ZENTEC'),
    ('2026-06-01 09:45:00', 'ACMESOLAR'),
    ('2026-06-01 10:00:00', 'WOCKPHARMA'),
    ('2026-06-01 10:30:00', 'WELSPUNLIV'),
    ('2026-06-02 10:00:00', 'ANANTRAJ'),
    ('2026-06-02 10:30:00', 'SONATSOFTW'),
    ('2026-06-03 10:00:00', 'TITAGARH'),
    ('2026-06-03 10:30:00', 'MRPL'),
    ('2026-06-04 10:00:00', 'AIAENG'),
    ('2026-06-04 10:45:00', 'TITAGARH'),
    ('2026-06-05 10:00:00', 'GODIGIT'),
    ('2026-06-05 10:00:00', 'IKS'),
    ('2026-06-05 10:00:00', 'CPPLUS'),
    ('2026-06-05 10:30:00', 'JYOTICNC'),
    ('2026-06-05 10:45:00', 'CHOLAFIN'),
    ('2026-06-05 10:45:00', 'CARBORUNIV'),
    ('2026-06-05 10:45:00', 'AUBANK'),
    ('2026-06-08 10:00:00', 'PFIZER'),
    ('2026-06-09 10:30:00', 'JAINREC'),
    ('2026-06-10 09:45:00', 'IDBI'),
    ('2026-06-11 10:00:00', 'BALRAMCHIN'),
    ('2026-06-11 10:45:00', 'AEGISVOPAK'),
    ('2026-06-12 09:45:00', 'KPITTECH'),
    ('2026-06-16 10:30:00', 'CESC'),
    ('2026-06-18 10:00:00', 'VTL'),
    ('2026-06-18 10:00:00', 'ITI'),
    ('2026-06-18 10:00:00', 'JSWENERGY'),
    ('2026-06-18 10:30:00', 'M&MFIN'),
    ('2026-06-19 10:00:00', 'TARIL'),
    ('2026-06-19 10:00:00', 'EMCURE'),
    ('2026-06-19 10:45:00', 'ABCAPITAL'),
    ('2026-06-22 09:45:00', 'TRENT'),
    ('2026-06-22 10:00:00', 'RITES'),
    ('2026-06-22 10:00:00', 'GRSE'),
    ('2026-06-22 10:30:00', 'VEDL'),
    ('2026-06-23 10:00:00', 'KIRLOSENG'),
    ('2026-06-23 10:30:00', 'JYOTICNC'),
    ('2026-06-25 09:45:00', 'SHRIRAMFIN'),
    ('2026-06-25 10:30:00', 'GODFRYPHLP'),
    ('2026-06-25 10:30:00', 'JSWCEMENT'),
    ('2026-06-29 10:45:00', 'APTUS'),
    ('2026-07-01 09:45:00', 'TARIL'),
    ('2026-07-01 10:00:00', 'JUBLINGREA'),
    ('2026-07-02 10:45:00', 'MAPMYINDIA'),
    ('2026-07-03 10:30:00', 'NUVAMA'),
    ('2026-07-06 10:00:00', 'SPLPETRO'),
    ('2026-07-06 10:00:00', 'GRAPHITE'),
    ('2026-07-08 09:45:00', 'ONGC'),
    ('2026-07-08 10:00:00', 'PFIZER'),
    ('2026-07-08 10:00:00', 'CARTRADE'),
    ('2026-07-09 10:00:00', 'WELCORP'),
    ('2026-07-09 10:45:00', 'GODREJIND'),
    ('2026-07-09 10:45:00', 'SIGNATURE'),
    ('2026-07-10 10:30:00', 'COHANCE'),
    ('2026-07-10 10:30:00', 'DOMS'),
    ('2026-07-13 10:00:00', 'ONGC'),
    ('2026-07-13 10:00:00', 'AJANTPHARM'),
    ('2026-07-13 10:30:00', 'CANHLIFE'),
    ('2026-07-15 10:00:00', 'DEEPAKFERT'),
    ('2026-07-15 10:00:00', 'POONAWALLA'),
    ('2026-07-15 10:30:00', 'AARTIIND'),
    ('2026-07-16 09:45:00', 'HDBFS'),
    ('2026-07-16 10:00:00', 'KAYNES'),
    ('2026-07-16 10:00:00', 'EMMVEE'),
    ('2026-07-16 10:30:00', 'DEEPAKFERT'),
    ('2026-07-17 10:30:00', 'BALRAMCHIN'),
    ('2026-07-20 10:30:00', 'KAJARIACER'),
    ('2026-07-22 09:45:00', 'HINDCOPPER'),
    ('2026-07-22 10:00:00', 'ANANTRAJ'),
    ('2026-07-23 10:00:00', 'NATIONALUM'),
    ('2026-07-24 10:00:00', 'NUVAMA'),
    ('2026-07-27 09:45:00', 'HINDZINC'),
    ('2026-07-27 10:30:00', 'ABREL'),
    ('2026-07-29 09:45:00', 'NETWEB'),
    ('2026-07-30 10:00:00', 'REDINGTON'),
    ('2026-07-30 10:30:00', 'IIFL'),
    ('2026-07-31 10:00:00', 'AARTIIND'),
    ('2026-08-04 09:45:00', 'GESHIP'),
    ('2026-08-04 10:00:00', 'GODFRYPHLP'),
    ('2026-08-04 10:00:00', 'CASTROLIND'),
    ('2026-08-05 10:30:00', 'POONAWALLA'),
    ('2026-08-05 10:30:00', 'PNBHOUSING'),
    ('2026-08-05 10:30:00', 'BIKAJI'),
    ('2026-08-06 10:00:00', 'GAIL'),
    ('2026-08-06 10:30:00', 'COCHINSHIP'),
    ('2026-08-07 10:30:00', 'MPHASIS'),
    ('2026-08-10 10:00:00', 'CONCORDBIO'),
    ('2026-08-11 10:00:00', 'JWL'),
    ('2026-08-11 10:30:00', 'KPRMILL'),
    ('2026-08-13 09:45:00', 'LENSKART'),
    ('2026-08-13 09:45:00', 'GROWW'),
    ('2026-08-13 10:00:00', 'BDL'),
    ('2026-08-13 10:30:00', 'CYIENT'),
    ('2026-08-13 10:30:00', 'BHARTIARTL'),
    ('2026-08-13 10:45:00', 'SUNTV'),
    ('2026-08-14 10:00:00', 'HONASA'),
    ('2026-08-17 10:00:00', 'BALRAMCHIN'),
    ('2026-08-18 10:30:00', 'MINDACORP'),
    ('2026-08-19 10:00:00', 'CHENNPETRO'),
    ('2026-08-19 10:00:00', 'GESHIP'),
    ('2026-08-20 09:45:00', 'INDGN'),
    ('2026-08-21 10:00:00', 'HINDCOPPER'),
    ('2026-08-21 10:00:00', 'JBMA'),
    ('2026-08-21 10:00:00', 'RAILTEL'),
    ('2026-08-24 09:45:00', 'NTPCGREEN'),
    ('2026-08-27 09:45:00', 'BAJFINANCE'),
    ('2026-08-27 10:00:00', 'CGPOWER'),
    ('2026-08-27 10:00:00', 'GVT&D'),
    ('2026-08-27 10:00:00', 'APLAPOLLO'),
    ('2026-08-27 10:30:00', 'NCC'),
    ('2026-08-27 10:30:00', 'SRF'),
    ('2026-08-27 10:30:00', 'PINELABS'),
    ('2026-08-28 10:00:00', 'MPHASIS'),
    ('2026-08-28 10:00:00', 'KPITTECH'),
    ('2026-08-28 10:00:00', 'HEXT'),
    ('2026-08-28 10:30:00', 'TEJASNET'),
    ('2026-08-28 10:45:00', 'TATAINVEST'),
]


for _, stock_symbol in stocks_info:
    stock_with_suffix = f"{stock_symbol}-EQ"
    if stock_with_suffix not in symbols_array:  # Avoid duplicates
        symbols_array.append(stock_with_suffix)

symbols_array.sort()  # Sort alphabetically

print(f"✓ Extracted {len(symbols_array)} unique stocks with -EQ suffix")
print()
print("Symbols to download:")
for i, symbol in enumerate(symbols_array, 1):
    print(f"  {i}. {symbol}")
print()

# ==================== STEP 2: BROKER API LOGIN ====================
print("=" * 80)
print("🔐 STEP 2: INITIALIZING BROKER API")
print("=" * 80)
print()




# ==================== STEP 3: PREPARE DOWNLOAD PARAMETERS ====================
print("=" * 80)
print("📅 STEP 3: PREPARING DATA DOWNLOAD PARAMETERS")
print("=" * 80)

# Calculate date range (4 months back)
now = datetime.now()
start_date = now - relativedelta(months=6)
start_date = start_date.replace(hour=0, minute=0, second=0, microsecond=0)
start_timestamp = start_date.timestamp()

print(f"Start date: {datetime.fromtimestamp(start_timestamp)}")
print(f"End date: {now}")
print(f"Data period: 4 months")
print(f"Exchange: NSE")
print(f"Interval: 1 minute")
print()

# ==================== STEP 4: DATA OUTPUT DIRECTORY ====================
output_directory = r'D:\AlgoRepo\ShoonyaAPI_Code\Testing_Use\Stocks_DATA'
print(f"Output directory: {output_directory}")
print()

# Create directory if it doesn't exist
import os
os.makedirs(output_directory, exist_ok=True)
print(f"✓ Directory ready")
print()

# ==================== STEP 5: DOWNLOAD HISTORICAL DATA ====================
print("=" * 80)
print("📥 STEP 5: DOWNLOADING HISTORICAL DATA")
print("=" * 80)
print()

print("Clearing old files...")
deleted_count = 0
failed_count = 0
for file in os.listdir(output_directory):
    file_path = os.path.join(output_directory, file)
    if os.path.isfile(file_path):
        try:
            os.remove(file_path)
            deleted_count += 1
        except PermissionError as e:
            print(f"⚠️  Could not delete {file} (file is locked). Skipping...")
            failed_count += 1
            
if deleted_count > 0:
    print(f"✓ Deleted {deleted_count} old files")
if failed_count > 0:
    print(f"⚠️  {failed_count} files could not be deleted (locked/in use)")
print()

successful_downloads = []
failed_downloads = []

for idx, stock in enumerate(symbols_array, 1):
    try:
        print(f"[{idx}/{len(symbols_array)}] Downloading {stock}...", end=" ")
        
        # Fetch the data from the API
        ret = api.get_time_price_series(exchange='NSE', token=stock, starttime=start_timestamp, interval=1)
        
        if ret and len(ret) > 0:
            # Convert the response into a DataFrame
            df_stock = pd.DataFrame(ret)
            
            # Save the DataFrame to an Excel file
            output_file_path = f'{output_directory}\\{stock}.xlsx'
            df_stock.to_excel(output_file_path, index=False)
            
            print(f"✓ ({len(df_stock)} rows saved)")
            successful_downloads.append(stock)
            
        else:
            print(f"✗ (No data returned)")
            failed_downloads.append(stock)
            
    except Exception as e:
        print(f"✗ (Error: {str(e)[:50]})")
        failed_downloads.append(stock)
        logging.error(f"Error downloading {stock}: {e}")

print()

# ==================== STEP 6: DOWNLOAD SUMMARY ====================
print("=" * 80)
print("✅ DOWNLOAD COMPLETE - SUMMARY REPORT")
print("=" * 80)
print()

print(f"📊 STATISTICS:")
print(f"   Total stocks: {len(symbols_array)}")
print(f"   Successfully downloaded: {len(successful_downloads)}")
print(f"   Failed downloads: {len(failed_downloads)}")
success_rate = (len(successful_downloads)/len(symbols_array)*100) if len(symbols_array) > 0 else 0
print(f"   Success rate: {success_rate:.1f}%")
print()

if successful_downloads:
    print(f"✓ SUCCESSFUL DOWNLOADS ({len(successful_downloads)}):")
    for i, stock in enumerate(successful_downloads, 1):
        print(f"   {i}. {stock}")
    print()

if failed_downloads:
    print(f"✗ FAILED DOWNLOADS ({len(failed_downloads)}):")
    for i, stock in enumerate(failed_downloads, 1):
        print(f"   {i}. {stock}")
    print()

print("=" * 80)
print(f"📁 Files saved to: {output_directory}")
print(f"🎯 All data ready for backtesting!")
print("=" * 80)



In [6]:
###############################################################################

#3

#this is code whre you need to paste in ('13-09-2024 10:00', 'MANAPPURAM'), this way and make sure all 
# the stocks are placed in folder whih will give the output with backtest .

################################################################################ 



import pandas as pd
import os

# Set the directory where the Excel sheets are stored
excel_directory = 'D:\\AlgoRepo\\ShoonyaAPI_Code\\Testing_Use\\Stocks_DATA'

# Stock info in format: (entry time, stock symbol)
stocks_info = [
    ('2026-05-19 10:00:00', 'AEGISLOG'),
    ('2026-05-19 10:00:00', 'INTELLECT'),
    ('2026-05-19 10:00:00', 'SWIGGY'),
    ('2026-05-19 10:30:00', 'DEEPAKFERT'),
    ('2026-05-19 10:30:00', 'KPIL'),
    ('2026-05-19 10:45:00', 'STARHEALTH'),
    ('2026-05-21 10:00:00', 'GODFRYPHLP'),
    ('2026-05-21 10:00:00', 'GRAPHITE'),
    ('2026-05-21 10:00:00', 'DATAPATTNS'),
    ('2026-05-21 10:30:00', 'ADANIPOWER'),
    ('2026-05-22 09:45:00', 'DALBHARAT'),
    ('2026-05-22 10:00:00', 'TRITURBINE'),
    ('2026-05-22 10:00:00', 'SBILIFE'),
    ('2026-05-27 10:30:00', 'THERMAX'),
    ('2026-05-27 10:30:00', 'AIAENG'),
    ('2026-05-27 10:30:00', 'ZENTEC'),
    ('2026-06-01 09:45:00', 'ACMESOLAR'),
    ('2026-06-01 10:00:00', 'WOCKPHARMA'),
    ('2026-06-01 10:30:00', 'WELSPUNLIV'),
    ('2026-06-02 10:00:00', 'ANANTRAJ'),
    ('2026-06-02 10:30:00', 'SONATSOFTW'),
    ('2026-06-03 10:00:00', 'TITAGARH'),
    ('2026-06-03 10:30:00', 'MRPL'),
    ('2026-06-04 10:00:00', 'AIAENG'),
    ('2026-06-04 10:45:00', 'TITAGARH'),
    ('2026-06-05 10:00:00', 'GODIGIT'),
    ('2026-06-05 10:00:00', 'IKS'),
    ('2026-06-05 10:00:00', 'CPPLUS'),
    ('2026-06-05 10:30:00', 'JYOTICNC'),
    ('2026-06-05 10:45:00', 'CHOLAFIN'),
    ('2026-06-05 10:45:00', 'CARBORUNIV'),
    ('2026-06-05 10:45:00', 'AUBANK'),
    ('2026-06-08 10:00:00', 'PFIZER'),
    ('2026-06-09 10:30:00', 'JAINREC'),
    ('2026-06-10 09:45:00', 'IDBI'),
    ('2026-06-11 10:00:00', 'BALRAMCHIN'),
    ('2026-06-11 10:45:00', 'AEGISVOPAK'),
    ('2026-06-12 09:45:00', 'KPITTECH'),
    ('2026-06-16 10:30:00', 'CESC'),
    ('2026-06-18 10:00:00', 'VTL'),
    ('2026-06-18 10:00:00', 'ITI'),
    ('2026-06-18 10:00:00', 'JSWENERGY'),
    ('2026-06-18 10:30:00', 'M&MFIN'),
    ('2026-06-19 10:00:00', 'TARIL'),
    ('2026-06-19 10:00:00', 'EMCURE'),
    ('2026-06-19 10:45:00', 'ABCAPITAL'),
    ('2026-06-22 09:45:00', 'TRENT'),
    ('2026-06-22 10:00:00', 'RITES'),
    ('2026-06-22 10:00:00', 'GRSE'),
    ('2026-06-22 10:30:00', 'VEDL'),
    ('2026-06-23 10:00:00', 'KIRLOSENG'),
    ('2026-06-23 10:30:00', 'JYOTICNC'),
    ('2026-06-25 09:45:00', 'SHRIRAMFIN'),
    ('2026-06-25 10:30:00', 'GODFRYPHLP'),
    ('2026-06-25 10:30:00', 'JSWCEMENT'),
    ('2026-06-29 10:45:00', 'APTUS'),
    ('2026-07-01 09:45:00', 'TARIL'),
    ('2026-07-01 10:00:00', 'JUBLINGREA'),
    ('2026-07-02 10:45:00', 'MAPMYINDIA'),
    ('2026-07-03 10:30:00', 'NUVAMA'),
    ('2026-07-06 10:00:00', 'SPLPETRO'),
    ('2026-07-06 10:00:00', 'GRAPHITE'),
    ('2026-07-08 09:45:00', 'ONGC'),
    ('2026-07-08 10:00:00', 'PFIZER'),
    ('2026-07-08 10:00:00', 'CARTRADE'),
    ('2026-07-09 10:00:00', 'WELCORP'),
    ('2026-07-09 10:45:00', 'GODREJIND'),
    ('2026-07-09 10:45:00', 'SIGNATURE'),
    ('2026-07-10 10:30:00', 'COHANCE'),
    ('2026-07-10 10:30:00', 'DOMS'),
    ('2026-07-13 10:00:00', 'ONGC'),
    ('2026-07-13 10:00:00', 'AJANTPHARM'),
    ('2026-07-13 10:30:00', 'CANHLIFE'),
    ('2026-07-15 10:00:00', 'DEEPAKFERT'),
    ('2026-07-15 10:00:00', 'POONAWALLA'),
    ('2026-07-15 10:30:00', 'AARTIIND'),
    ('2026-07-16 09:45:00', 'HDBFS'),
    ('2026-07-16 10:00:00', 'KAYNES'),
    ('2026-07-16 10:00:00', 'EMMVEE'),
    ('2026-07-16 10:30:00', 'DEEPAKFERT'),
    ('2026-07-17 10:30:00', 'BALRAMCHIN'),
    ('2026-07-20 10:30:00', 'KAJARIACER'),
    ('2026-07-22 09:45:00', 'HINDCOPPER'),
    ('2026-07-22 10:00:00', 'ANANTRAJ'),
    ('2026-07-23 10:00:00', 'NATIONALUM'),
    ('2026-07-24 10:00:00', 'NUVAMA'),
    ('2026-07-27 09:45:00', 'HINDZINC'),
    ('2026-07-27 10:30:00', 'ABREL'),
    ('2026-07-29 09:45:00', 'NETWEB'),
    ('2026-07-30 10:00:00', 'REDINGTON'),
    ('2026-07-30 10:30:00', 'IIFL'),
    ('2026-07-31 10:00:00', 'AARTIIND'),
    ('2026-08-04 09:45:00', 'GESHIP'),
    ('2026-08-04 10:00:00', 'GODFRYPHLP'),
    ('2026-08-04 10:00:00', 'CASTROLIND'),
    ('2026-08-05 10:30:00', 'POONAWALLA'),
    ('2026-08-05 10:30:00', 'PNBHOUSING'),
    ('2026-08-05 10:30:00', 'BIKAJI'),
    ('2026-08-06 10:00:00', 'GAIL'),
    ('2026-08-06 10:30:00', 'COCHINSHIP'),
    ('2026-08-07 10:30:00', 'MPHASIS'),
    ('2026-08-10 10:00:00', 'CONCORDBIO'),
    ('2026-08-11 10:00:00', 'JWL'),
    ('2026-08-11 10:30:00', 'KPRMILL'),
    ('2026-08-13 09:45:00', 'LENSKART'),
    ('2026-08-13 09:45:00', 'GROWW'),
    ('2026-08-13 10:00:00', 'BDL'),
    ('2026-08-13 10:30:00', 'CYIENT'),
    ('2026-08-13 10:30:00', 'BHARTIARTL'),
    ('2026-08-13 10:45:00', 'SUNTV'),
    ('2026-08-14 10:00:00', 'HONASA'),
    ('2026-08-17 10:00:00', 'BALRAMCHIN'),
    ('2026-08-18 10:30:00', 'MINDACORP'),
    ('2026-08-19 10:00:00', 'CHENNPETRO'),
    ('2026-08-19 10:00:00', 'GESHIP'),
    ('2026-08-20 09:45:00', 'INDGN'),
    ('2026-08-21 10:00:00', 'HINDCOPPER'),
    ('2026-08-21 10:00:00', 'JBMA'),
    ('2026-08-21 10:00:00', 'RAILTEL'),
    ('2026-08-24 09:45:00', 'NTPCGREEN'),
    ('2026-08-27 09:45:00', 'BAJFINANCE'),
    ('2026-08-27 10:00:00', 'CGPOWER'),
    ('2026-08-27 10:00:00', 'GVT&D'),
    ('2026-08-27 10:00:00', 'APLAPOLLO'),
    ('2026-08-27 10:30:00', 'NCC'),
    ('2026-08-27 10:30:00', 'SRF'),
    ('2026-08-27 10:30:00', 'PINELABS'),
    ('2026-08-28 10:00:00', 'MPHASIS'),
    ('2026-08-28 10:00:00', 'KPITTECH'),
    ('2026-08-28 10:00:00', 'HEXT'),
    ('2026-08-28 10:30:00', 'TEJASNET'),
    ('2026-08-28 10:45:00', 'TATAINVEST'),
]

# Result storage
results = []

# Initial investment per stock
initial_investment = 200000

# Define 15:15 time for cutting positions
cutoff_time = pd.to_datetime('2026-03-25 15:15', format='%Y-%m-%d %H:%M')

# Iterate over each stock and its respective entry time
for entry_time_str, stock_name in stocks_info:

    # Files are named with -EQ suffix (e.g., MRPL-EQ.xlsx)
    excel_file_name = f"{stock_name}-EQ.xlsx"
    excel_file_path = os.path.join(excel_directory, excel_file_name)

    try:
        # Load stock data
        stock_data = pd.read_excel(excel_file_path, usecols=['time', 'intc'])
        stock_data['time'] = pd.to_datetime(stock_data['time'], format='%d-%m-%Y %H:%M:%S', errors='coerce')

        # Sort the data, ensuring latest data is processed correctly
        stock_data.sort_values(by='time', ascending=True, inplace=True)

        # Convert entry_time_str to timestamp
        entry_time = pd.to_datetime(entry_time_str, format='%Y-%m-%d %H:%M:%S')

        # Add 2 minutes to the entry time for testing
        testing_start_time = entry_time + pd.Timedelta(minutes=2)

        # Use .iloc[0] to get first row after testing_start_time
        entry_rows = stock_data[stock_data['time'] >= testing_start_time]
        
        if not entry_rows.empty:
            entry_row = entry_rows.iloc[0]
            entry_price = entry_row['intc']
            qty = int(initial_investment / entry_price)  # Calculate quantity of stocks

            # # For SHORT SELLING
            # profit_target = entry_price * 0.988  # Target at 0.7% LOWER (profit on short)
            # stop_loss = entry_price * 1.006    # Stop loss at 1.5% HIGHER (protect from rise)
            # For LONG SELLING
            profit_target = entry_price * 0.988 #get at 0.7% LOWER (profit on short)
            stop_loss = entry_price * 1.006 # Stop loss at 1.5% HIGHER (ssprotect from rise)

            # Filter subsequent data (after the entry row)
            subsequent_data = stock_data[stock_data['time'] > entry_row['time']]

            stop_or_tgt_hit = False
            for index, row in subsequent_data.iterrows():
                current_price = row['intc']
                current_time = row['time']

                # Check profit target first (price BELOW entry = profit on short)
                if current_price < profit_target:
                    profit_loss_amount = ((entry_price - current_price) * qty)
                    results.append([stock_name, entry_time, current_time, current_price, 'Profit', profit_loss_amount])
                    print(f"Profit target hit at {current_time}: {current_price:.2f}, Profit: {profit_loss_amount:.2f}")
                    stop_or_tgt_hit = True
                    break

                # Check stop loss (price ABOVE entry = loss on short)
                elif current_price > stop_loss:
                    profit_loss_amount = ((entry_price - current_price) * qty)
                    results.append([stock_name, entry_time, current_time, current_price, 'Loss', profit_loss_amount])
                    print(f"Stop loss hit at {current_time}: {current_price:.2f}, Loss: {profit_loss_amount:.2f}")
                    stop_or_tgt_hit = True
                    break

            # Ensure the trade is closed at cutoff time if no stop-loss or target is hit
            cutoff_rows = stock_data[stock_data['time'] >= cutoff_time]
            if not stop_or_tgt_hit and not cutoff_rows.empty:
                cutoff_price = cutoff_rows.iloc[0]['intc']
                profit_loss_amount = ((entry_price - cutoff_price) * qty)
                status = 'Profit' if cutoff_price < entry_price else 'Loss'
                results.append([stock_name, entry_time, cutoff_time, cutoff_price, f'Cut at 15:15 ({status})', profit_loss_amount])
                print(f"No SL or TGT hit. Position closed at {cutoff_time}: {cutoff_price:.2f}, PnL: {profit_loss_amount:.2f}")
            elif not stop_or_tgt_hit and cutoff_rows.empty:
                print(f"No data found for {stock_name} at the cutoff time (15:15).")

        else:
            print(f"No entry found for {stock_name} at {testing_start_time}")

    except Exception as e:
        print(f"An error occurred for {stock_name}: {e}")

# Create a DataFrame for the results
results_df = pd.DataFrame(results, columns=['Stock Name', 'Entry Time', 'Hit/Exit Time', 'Price', 'Status', 'Profit/Loss Amount'])

# Save the results to an Excel file
output_file_path = 'C:\\Users\\omkar\\Downloads\\stock_results_New_0930TO11_spec.xlsx'
results_df.to_excel(output_file_path, index=False)

print(f"Results saved to {output_file_path}")
print(f"Total results: {len(results)}")

# ==================== RESULTS SUMMARY ====================
print("\n" + "=" * 80)
print("📊 BACKTEST RESULTS SUMMARY")
print("=" * 80)
print()

total_profit_loss = results_df['Profit/Loss Amount'].sum()
winning_trades = len(results_df[results_df['Profit/Loss Amount'] > 0])
losing_trades = len(results_df[results_df['Profit/Loss Amount'] < 0])
total_trades = len(results_df)
win_rate = (winning_trades / total_trades * 100) if total_trades > 0 else 0

print(f"💰 TOTAL PROFIT/LOSS: Rs. {total_profit_loss:,.2f}")
print(f"✅ WINNING TRADES: {winning_trades}")
print(f"❌ LOSING TRADES: {losing_trades}")
print(f"📈 TOTAL TRADES: {total_trades}")
print(f"🎯 WIN RATE: {win_rate:.1f}%")
print()
print("=" * 80)
print(f"📁 Files saved to: {output_file_path}")
print("=" * 80)

print(f"\nResults saved to {output_file_path}")


Stop loss hit at 2026-05-19 10:11:00: 703.50, Loss: -1937.25
Stop loss hit at 2026-05-19 10:37:00: 717.60, Loss: -1498.00
Stop loss hit at 2026-05-19 12:32:00: 261.65, Loss: -2945.00
Profit target hit at 2026-05-19 14:48:00: 1320.30, Profit: 2413.80
Profit target hit at 2026-05-19 14:36:00: 1252.30, Profit: 2402.10
Stop loss hit at 2026-05-19 14:12:00: 512.75, Loss: -1215.20
Profit target hit at 2026-05-21 10:55:00: 2317.00, Profit: 2550.00
Profit target hit at 2026-05-21 11:39:00: 753.00, Profit: 2528.30
Profit target hit at 2026-05-21 11:18:00: 3903.20, Profit: 2375.00
Profit target hit at 2026-05-21 12:01:00: 222.17, Profit: 2513.04
Stop loss hit at 2026-05-22 10:15:00: 1788.10, Loss: -1422.40
Stop loss hit at 2026-05-22 10:52:00: 731.05, Loss: -1540.00
Stop loss hit at 2026-05-25 14:17:00: 1895.40, Loss: -1261.40
Profit target hit at 2026-05-27 12:07:00: 4520.40, Profit: 2369.30
Stop loss hit at 2026-05-27 10:40:00: 4325.80, Loss: -1416.80
Stop loss hit at 2026-05-27 11:17:00: 1673